In [ ]:
%matplotlib qt5

import hyperspy.api as hs
import pyxem as pxm
import numpy as np
import orix
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import matplotlib as mpl
import copy
import pickle

from pathlib import Path
from diffpy.structure import Atom, Lattice, Structure
from diffsims.generators.simulation_generator import SimulationGenerator
from orix.vector import Miller, Vector3d
from orix.plot import IPFColorKeyTSL
from matplotlib.lines import Line2D
from dataclasses import dataclass

In [ ]:
## Paths used for loading and saving data
path = Path("D:\\Datasets\\SPED_tilt_series\\SPED_tilt_series")
path_calibrated = Path("D:\\Datasets\\SPED_tilt_series\\series_calibrated")

path_target = Path("D:\\Datasets\\SPED_tilt_series\\target_areas")
path_reference = Path("D:\\Datasets\\SPED_tilt_series\\reference_areas")

path_target_masked = Path("D:\\Datasets\\SPED_tilt_series\\target_areas_masked")
path_reference_masked = Path("D:\\Datasets\\SPED_tilt_series\\reference_areas_masked")

target_files = sorted(path_target.glob("*.hspy"))
reference_files = sorted(path_reference.glob("*.hspy"))

target_files_masked = sorted(path_target_masked.glob("*.hspy"))
reference_files_masked = sorted(path_reference_masked.glob("*.hspy"))

## Preprocessing

In [ ]:
# The scaling factor used was determined by manually calibrating one of the datasets and extracting the scale factor from the calibrated data. #
def preprocess_data(path, path_calibrated, path_target, path_reference, scale_factor):
    files = sorted(path.glob("*.hspy"))
    for i in files:
        filename = i.stem
        number = filename.rsplit("_", 1)[-1]

        ## Calibrate the data ##
        load_i = hs.load(i)
        load_i.axes_manager[2].scale = scale_factor
        load_i.axes_manager[3].scale = scale_factor

        shifts = load_i.get_direct_beam_position(method="blur", sigma=5, half_square_width=10)
        linear_shifts = shifts.get_linear_plane()
        load_i_centered = load_i.center_direct_beam(shifts=linear_shifts, inplace=False)


        ## Select region of interest ##
        reference_rectangle = hs.roi.RectangularROI(left=0, top=0, right=10, bottom=10)
        target_rectangle = hs.roi.RectangularROI(left=0, top=0, right=10, bottom=10)

        load_i_centered.plot()

        reference_roi = reference_rectangle.interactive(load_i_centered)
        target_roi = target_rectangle.interactive(load_i_centered, color='C1')
        
        plt.show(block=True)

        reference_roi.save(path_reference / f"SPED_tilt_series_reference_{number}.hspy")
        target_roi.save(path_target / f"SPED_tilt_series_target_{number}.hspy")
        load_i_centered.save(path_calibrated / f"SPED_tilt_series_calibrated_{number}.hspy")

## Masking

In [ ]:
def masking_function(path_target, path_reference, path_target_masked, path_reference_masked):
    files_target = sorted(path_target.glob("*.hspy"))
    files_reference = sorted(path_reference.glob("*.hspy"))

    for number, j in enumerate(zip(files_target, files_reference)):
        
        load_target = hs.load(j[0])
        load_reference = hs.load(j[1])

        peaks_target = load_target.get_diffraction_vectors(min_distance=18, threshold_abs=0.1)
        peaks_reference = load_reference.get_diffraction_vectors(min_distance=18, threshold_abs=0.1)

        mask_target = peaks_target.to_mask(disk_r = 7)
        mask_reference = peaks_reference.to_mask(disk_r = 7)

        masked_target = load_target * mask_target
        masked_reference = load_reference * mask_reference

        masked_target.save(path_target_masked / f"target_masked_{number}.hspy")
        masked_reference.save(path_reference_masked / f"reference_masked_{number}.hspy")
    
masking_function(path_target, path_reference, path_target_masked, path_reference_masked)

## Simulation

In [ ]:
# Define the crystal structure for aluminum (Al)
a = 4.05
atoms = [Atom('Al', [0, 0, 0]), Atom('Al', [0.5, 0.5, 0]), Atom('Al', [0.5, 0, 0.5]), Atom('Al', [0, 0.5, 0.5])]
lattice = Lattice(a, a, a, 90, 90, 90)
phase = orix.crystal_map.Phase(name='Al', space_group=225, structure=Structure(atoms, lattice))

angular_resolution = 0.5
grid = orix.sampling.get_sample_reduced_fundamental(angular_resolution, point_group=phase.point_group)
orientations = orix.quaternion.Orientation(grid, symmetry=phase.point_group)

simgen = SimulationGenerator(precession_angle=1., minimum_intensity=1e-5)
simulations = simgen.calculate_diffraction2d(phase=phase, rotation=grid, reciprocal_radius=1.5, with_direct_beam=False, max_excitation_error=0.01)

# Create IPF color keys for x, y, and z directions
key_x = IPFColorKeyTSL(phase.point_group, Vector3d.xvector())
key_y = IPFColorKeyTSL(phase.point_group, Vector3d.yvector())
key_z = IPFColorKeyTSL(phase.point_group, Vector3d.zvector())

In [ ]:
@dataclass
class SPEDResult:
    results: object
    correlations: object
    best_single_phase: object   
    
    def ipf_correlation(self):
        template_indices_i = (self.results.data[:, 0]).astype('int16')
        orientations_i = orientations[template_indices_i]
        
        fig = plt.figure()
        ax = fig.add_subplot(111, projection="ipf", symmetry=phase.point_group)
        ax.scatter(orientations_i, c=self.correlations, cmap='inferno')
        ax.scatter(orientations_i[0], c="r", marker="x")
        
        fig_dp, ax_dp = plt.subplots()
        first = template_indices_i[0]
        
        sim_i = simulations.irot[first]
        pat = sim_i.get_diffraction_pattern(
            shape=(256, 256),        # pick your detector size
            sigma=2,                 # spot size in pixels
            calibration=0.01,        # real->pixel scale (tune to match your data)
            direct_beam_position=(128, 128),
            normalize=True,
        )
        ax_dp.imshow(pat, vmax=1, cmap='magma')

    def ipf_matches(self):
        fig = self.single_phase.scatter(projection="ipf", return_figure=True)
        plt.show()

In [ ]:
def template(target_data, ref_data):
    ref_data.change_dtype("float32")
    target_data.change_dtype("float32")

    ny, nx = target_data.axes_manager.signal_shape
    center = (ny/2 - 0.5, nx/2 - 0.5)
    
    ref_data.calibration(center=center)
    target_data.calibration(center=center)
    
    target_azi = target_data.get_azimuthal_integral2d(npt=256)
    ref_azi = ref_data.get_azimuthal_integral2d(npt=256)
    
    target_results = target_azi.get_orientation(simulations, n_best=grid.size, frac_keep=0.5)
    ref_results = ref_azi.get_orientation(simulations, n_best=grid.size, frac_keep=0.5)
    
    target_single_phase = target_results.to_single_phase_orientations()
    reference_single_phase = ref_results.to_single_phase_orientations()
    
    target_correlation = target_results.data[:, 1]
    reference_correlation = ref_results.data[:, 1]
    
    target = SPEDResult(
        results=target_results, 
        correlations=target_correlation, 
        best_single_phase=target_single_phase)
    
    reference = SPEDResult(
        results=ref_results, 
        correlations=reference_correlation, 
        best_single_phase=reference_single_phase)
    
    return target, reference

In [ ]:
def tilt_results():
    misorientation_array = np.empty((len(target_files_masked), 2), dtype=object)
    for i, files in enumerate(zip(target_files_masked, reference_files_masked)):
        
        target_data = hs.load(files[0])
        reference_data = hs.load(files[1])

        target_sum = target_data.sum() ** 0.5 - 5
        reference_mean = reference_data.sum() ** 0.5 - 5

        target_result, reference_result = template(target_sum, reference_mean)

        misorientation_array[i,0] = target_result 
        misorientation_array[i,1] = reference_result
    
    return misorientation_array

result_array = tilt_results()

In [ ]:
for i in range(len(result_array)):
    for j in range(2):
        obj = result_array[i, j]
        
        # force evaluation if needed
        obj.results = list(obj.results)
with open("tilt_results.pkl", "wb") as f:
    pickle.dump(result_array, f)

In [ ]:
with open("tilt_results.pkl", "rb") as f:
    result_array = pickle.load(f)

In [ ]:
def fixing(results, number_target=51, number_reference=35, angle_threshold=0.2):
    fixed_results = copy.deepcopy(results)

    for i in range(number_target, len(results)):
        index = 1
        angle = fixed_results[i-1][0].best_single_phase[0].angle_with(fixed_results[i][0].best_single_phase[0])
        if angle > angle_threshold:
            while angle > angle_threshold:
                if index >= len(fixed_results[i][0].best_single_phase):
                    print(f"Target dataset {i} has no suitable orientation within the angle threshold. Keeping index 0.")
                    break
                angle = fixed_results[i-1][0].best_single_phase[0].angle_with(fixed_results[i][0].best_single_phase[index])
                index += 1
            print(f"Target dataset {i} was changed from index 0 to index {index-1} with angle {angle} radians.")
            fixed_results[i][0].best_single_phase = fixed_results[i][0].best_single_phase[index-1]
    
    for j in range(number_target, -1, -1):
        index = 1
        angle = fixed_results[j+1][0].best_single_phase[0].angle_with(fixed_results[j][0].best_single_phase[0])
        if angle > angle_threshold:
            while angle > angle_threshold:
                if index >= len(fixed_results[j][0].best_single_phase):
                    print(f"Target dataset {j} has no suitable orientation within the angle threshold. Keeping index 0.")
                    break
                angle = fixed_results[j+1][0].best_single_phase[0].angle_with(fixed_results[j][0].best_single_phase[index])
                index += 1
            print(f"Target dataset {j} was changed from index 0 to index {index-1} with angle {angle} radians.")
            fixed_results[j][0].best_single_phase = fixed_results[j][0].best_single_phase[index-1]
    
    for k in range(number_reference, len(results)):
        index = 1
        angle = fixed_results[k-1][1].best_single_phase[0].angle_with(fixed_results[k][1].best_single_phase[0])
        if angle > angle_threshold:
            while angle > angle_threshold:
                if index >= len(fixed_results[k][1].best_single_phase):
                    print(f"Reference dataset {k} has no suitable orientation within the angle threshold. Keeping index 0.")
                    break
                angle = fixed_results[k-1][1].best_single_phase[0].angle_with(fixed_results[k][1].best_single_phase[index])
                index += 1
            print(f"Reference dataset {k} was changed from index 0 to index {index-1} with angle {angle} radians.")
            fixed_results[k][1].best_single_phase = fixed_results[k][1].best_single_phase[index-1]
    
    for l in range(number_reference, -1, -1):
        index = 1
        angle = fixed_results[l+1][1].best_single_phase[0].angle_with(fixed_results[l][1].best_single_phase[0])
        if angle > angle_threshold:
            while angle > angle_threshold:
                if index >= len(fixed_results[l][1].best_single_phase):
                    print(f"Reference dataset {l} has no suitable orientation within the angle threshold. Keeping index 0.")
                    break
                angle = fixed_results[l+1][1].best_single_phase[0].angle_with(fixed_results[l][1].best_single_phase[index])
                index += 1
            print(f"Reference dataset {l} was changed from index 0 to index {index-1} with angle {angle} radians.")
            fixed_results[l][1].best_single_phase = fixed_results[l][1].best_single_phase[index-1]
    
    return fixed_results

fixed_result_array = fixing(result_array)

In [ ]:
def return_orix_orientation(result_array):
    misorientation = np.empty(60, dtype=object)
    orientation_target = np.empty(60, dtype=object)
    orientation_reference = np.empty(60, dtype=object)

    for number, i in enumerate(result_array):
        orientation_target[number] = i[0].best_single_phase
        orientation_reference[number] = i[1].best_single_phase

    quat_array = np.stack([j.data for j in orientation_target])
    orientation_target = orix.quaternion.Orientation(quat_array, symmetry=phase.point_group)

    quat_array = np.stack([k.data for k in orientation_reference])
    orientation_reference = orix.quaternion.Orientation(quat_array, symmetry=phase.point_group)

    return orientation_target, orientation_reference

orientation_target, orientation_reference = return_orix_orientation(result_array)

In [ ]:
def plot_ipf(orientation):
    c_scalar = np.arange(1, 61)

    fig = plt.figure()
    ax = fig.add_subplot(projection='ipf', symmetry=phase.point_group)
    sc = ax.scatter(orientation, c=c_scalar)

    # Create colorbar manually
    norm = mpl.colors.Normalize(vmin=c_scalar.min(), vmax=c_scalar.max())
    sm = mpl.cm.ScalarMappable(norm=norm)
    sm.set_array([])

    cbar = plt.colorbar(sm, ax=ax)
    cbar.set_label("Dataset number")
    plt.show()

plot_ipf(orientation_target)
plot_ipf(fixed_orientation_target)

In [ ]:
def main_function2(data_list, name):
    fig = go.Figure()
    
    rxyz = np.asarray(data_list.to_axes_angles().xyz)
    rx = rxyz[0].ravel()
    ry = rxyz[1].ravel()
    rz = rxyz[2].ravel()
    
    c_scalar = np.arange(len(rx))

    fig.add_trace(
    go.Scatter3d(
        x=rx, y=ry, z=rz,
        mode="markers+lines",
        marker=dict(
            color=c_scalar,              # 👈 THIS is the key
            colorscale="Viridis",        # choose colormap
            opacity=0.85,
            size=4,
            showscale=True,
            colorbar=dict(
                title="Datapoint index",
                x=-0.15,
                thickness=15,
                len=0.6
                )
            ),
        showlegend=True
        )
    )

    R = 3.4

    fig.update_layout(
        scene=dict(
            xaxis=dict(range=[-R, R], title="x"),
            yaxis=dict(range=[-R, R], title="y"),
            zaxis=dict(range=[-R, R], title="z"),
            aspectmode="cube"   # critical: equal scaling
        )
    )
    axis_lines = [
        # X-axis
        go.Scatter3d(
            x=[-R, R], y=[0, 0], z=[0, 0],
            mode="lines",
            line=dict(color="black", width=4),
            name="X axis",
            showlegend=False
        ),
        # Y-axis
        go.Scatter3d(
            x=[0, 0], y=[-R, R], z=[0, 0],
            mode="lines",
            line=dict(color="black", width=4),
            name="Y axis",
            showlegend=False
        ),
        # Z-axis
        go.Scatter3d(
            x=[0, 0], y=[0, 0], z=[-R, R],
            mode="lines",
            line=dict(color="black", width=4),
            name="Z axis",
            showlegend=False
        ),
    ]

    for trace in axis_lines:
        fig.add_trace(trace)

    fig.write_html(
    name,
    include_plotlyjs="cdn"
    )

main_function2(orientation_target, "orientation_target_fixed.html")